# Curso IA Commercial — Cuaderno 04: El Universo del Transformer en PyTorch

Este cuaderno interactivo complementa el **Cuaderno 04** del curso. Aquí implementamos desde cero en **PyTorch** los componentes nucleares de la arquitectura Transformer descrita en *"Attention Is All You Need"* (Vaswani et al., 2017).

**Módulos del Cuaderno:**
1. Demostración Matemática del Factor de Escala $1/\sqrt{d_k}$
2. Implementación de Scaled Dot-Product Attention con Máscara Causal
3. Módulo de Multi-Head Attention (MHA) Completo
4. Codificación Posicional Rotatoria (RoPE) vs Sinusoidal (APE)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo de cómputo: {device}")

## 1. Por qué escalamos por $1/\sqrt{d_k}$ (Varianza y Softmax)

Sean dos vectores aleatorios independientes $q, k \sim \mathcal{N}(0, 1)$ en $\mathbb{R}^{d_k}$.
El producto escalar es $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$.
- Cada término $q_i k_i$ tiene media $0$ y varianza $\text{Var}(q_i k_i) = \text{Var}(q_i)\text{Var}(k_i) = 1 \times 1 = 1$.
- Por linealidad de la varianza en variables independientes, $\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} 1 = d_k$.
- Para dimensionalidades grandes (ej. $d_k = 64$ o $128$), la varianza es $64$ o $128$, provocando valores extremos que saturan la función $\text{Softmax}$ hacia gradientes prácticamente nulos ($0$).
- Al dividir por $\sqrt{d_k}$, la varianza se normaliza a $\frac{d_k}{(\sqrt{d_k})^2} = 1$.

In [ ]:
d_k = 128
num_samples = 10000

q = torch.randn(num_samples, d_k)
k = torch.randn(num_samples, d_k)

unscaled_dot = (q * k).sum(dim=-1)
scaled_dot = unscaled_dot / math.sqrt(d_k)

print(f"Varianza sin escalar (esperada ~{d_k}): {unscaled_dot.var().item():.2f}")
print(f"Varianza escalada (esperada ~1.00):      {scaled_dot.var().item():.2f}")

plt.figure(figsize=(10, 4))
plt.hist(unscaled_dot.numpy(), bins=50, alpha=0.5, label=f'Sin escalar (Var={unscaled_dot.var():.1f})', color='salmon')
plt.hist(scaled_dot.numpy(), bins=50, alpha=0.7, label=f'Escalado 1/sqrt(d_k) (Var={scaled_dot.var():.1f})', color='royalblue')
plt.title("Distribución de Productos Escalares Q·K")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Autoatención Escalada (Scaled Dot-Product Attention)

$$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    # 1. Producto matricial Q * K^T -> (B, h, N, N)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    
    # 2. Aplicar máscara causal si existe
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # 3. Softmax sobre la última dimensión
    attn_weights = F.softmax(scores, dim=-1)
    
    # 4. Multiplicación por valores V -> (B, h, N, d_v)
    output = torch.matmul(attn_weights, v)
    return output, attn_weights

## 3. Módulo Multi-Head Attention (MHA) en PyTorch

Implementación modular completa con verificación de tensores en cada paso.

In [ ]:
class MultiHeadAttentionLab(nn.Module):
    def __init__(self, d_model=128, num_heads=4):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()
        
        # Proyecciones lineales y división en cabezas
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Calcular atención escalada
        attn_out, weights = scaled_dot_product_attention(q, k, v, mask)
        
        # Concatenar cabezas y mezclar
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.out_proj(attn_out)
        return output, weights

# Prueba de pase hacia adelante
B, N, d_model = 2, 6, 128
x_dummy = torch.randn(B, N, d_model)
mha = MultiHeadAttentionLab(d_model=d_model, num_heads=4)

out, weights = mha(x_dummy)
print(f"Input shape:   {x_dummy.shape}")
print(f"Output shape:  {out.shape}")
print(f"Weights shape: {weights.shape} -> (Batch, Heads, Seq_len, Seq_len)")
print("\n¡Bloque Multi-Head Attention validado con éxito en PyTorch!")